# 02 - Normalización y limpieza inicial

**Objetivo.** Normalizar textos, categorías, barrios, monedas, operaciones, fechas y valores inválidos; construir versiones preprocesadas por fuente/operación y una base consolidada con columnas canónicas.

La limpieza se realiza por dataset para no mezclar problemas específicos de una fuente con faltantes estructurales de otra.

In [88]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

REPO_ROOT: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026
RAW_DIR: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026\data\raw
PROCESSED_DIR: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026\data\processed


## Carga de archivos raw

In [89]:
INVALID_MARKERS = {
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'SIN DATOS', 'SIN_DATO','Sin Datos', 'sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

RAW_SOURCES = {
    'df_raw_meli_alq': {
        'descripcion': 'Mercado Libre - alquiler permanente',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler').glob('meli_alq_*.csv')),
    },
    'df_raw_meli_alq_temp': {
        'descripcion': 'Mercado Libre - alquiler temporario',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler temporario').glob('meli_alq_temp_*.csv')),
    },
    'df_raw_meli_vtas': {
        'descripcion': 'Mercado Libre - venta',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_ventas').glob('meli_vtas_*.csv')),
    },
    'df_raw_argenprop': {
        'descripcion': 'Argenprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'argenprop' / 'propiedades_argenprop.csv'],
    },
    'df_raw_zonaprop': {
        'descripcion': 'Zonaprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'zonaprop' / 'propiedades_zonaprop.csv'],
    },
    'df_raw_airbnb_norm': {
        'descripcion': 'Airbnb CABA normalizado mensual. Base prioritaria para analisis comparable',
        'paths': [RAW_DIR / 'airbnb' / 'airbnb_caba_normalizado_mensual.csv'],
    },
}


def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)


def load_many(paths, source_name):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        print(f'ATENCION: no se encontraron archivos para {source_name}')
        return pd.DataFrame()
    frames = []
    for path in existing:
        df = read_csv_safely(path)
        df['_archivo_origen'] = str(path.relative_to(REPO_ROOT))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

raw_dfs = {}
for df_name, spec in RAW_SOURCES.items():
    raw_dfs[df_name] = load_many(spec['paths'], df_name)
    globals()[df_name] = raw_dfs[df_name]
    print(f"{df_name}: {raw_dfs[df_name].shape} | {spec['descripcion']}")

df_raw_meli_alq: (11479, 61) | Mercado Libre - alquiler permanente
df_raw_meli_alq_temp: (6190, 61) | Mercado Libre - alquiler temporario
df_raw_meli_vtas: (44013, 61) | Mercado Libre - venta
df_raw_argenprop: (162, 96) | Argenprop - publicaciones extraidas
df_raw_zonaprop: (980, 96) | Zonaprop - publicaciones extraidas
df_raw_airbnb_norm: (9596, 61) | Airbnb CABA normalizado mensual. Base prioritaria para analisis comparable


## Funciones de limpieza, normalización y consolidación

In [90]:
CABA_BARRIOS = { 
    'agronomia':'agronomia', 'almagro':'almagro', 'balvanera':'balvanera', 'barracas':'barracas',
    'belgrano':'belgrano', 'boedo':'boedo', 'caballito':'caballito', 'chacarita':'chacarita',
    'coghlan':'coghlan', 'colegiales':'colegiales', 'constitucion':'constitucion', 'flores':'flores',
    'floresta':'floresta', 'la boca':'la boca', 'liniers':'liniers', 'mataderos':'mataderos',
    'monserrat':'monserrat', 'monte castro':'monte castro', 'nueva pompeya':'nueva pompeya',
    'nunez':'nunez', 'palermo':'palermo', 'parque avellaneda':'parque avellaneda',
    'parque chacabuco':'parque chacabuco', 'parque chas':'parque chas',
    'parque patricios':'parque patricios', 'paternal':'paternal', 'puerto madero':'puerto madero',
    'recoleta':'recoleta', 'retiro':'retiro', 'saavedra':'saavedra', 'san cristobal':'san cristobal',
    'san nicolas':'san nicolas', 'san telmo':'san telmo', 'velez sarsfield':'velez sarsfield',
    'versalles':'versalles', 'villa crespo':'villa crespo', 'villa del parque':'villa del parque',
    'villa devoto':'villa devoto', 'villa general mitre':'villa general mitre', 'villa lugano':'villa lugano',
    'villa luro':'villa luro', 'villa ortuzar':'villa ortuzar', 'villa pueyrredon':'villa pueyrredon',
    'villa real':'villa real', 'villa riachuelo':'villa riachuelo', 'villa santa rita':'villa santa rita',
    'villa soldati':'villa soldati', 'villa urquiza':'villa urquiza'
}

BARRIO_ALIASES = {#diccionario de normalizacion de barrios, segun tildes, articulos, abreviaturas, etc
    'nuñez': 'nunez', 'nunez': 'nunez', 'agronomía': 'agronomia',
    'san nicolás': 'san nicolas', 'san cristóbal': 'san cristobal',
    'vélez sársfield': 'velez sarsfield', 'villa gral. mitre': 'villa general mitre',
    'villa gral mitre': 'villa general mitre', 'palermo chico': 'palermo',
    'palermo hollywood': 'palermo', 'palermo soho': 'palermo', 'las cañitas': 'palermo',
    'belgrano chico': 'belgrano', 'barrio norte': 'recoleta',
    'capital federal': np.nan, 'ciudad autonoma de buenos aires': np.nan,
    'ciudad autónoma de buenos aires': np.nan, 'caba': np.nan,
}


def strip_accents(value):
    if pd.isna(value):
        return np.nan
    text = unicodedata.normalize('NFKD', str(value))
    return ''.join(ch for ch in text if not unicodedata.combining(ch))


def clean_text(value, lower=True):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ').strip()
    text = re.sub(r'\s+', ' ', text)
    if text in INVALID_MARKERS:
        return np.nan
    if lower:
        text = strip_accents(text).lower()
    return text

#funciones de normalizacion para campos problematicos
def normalize_barrio(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    text = BARRIO_ALIASES.get(text, text)
    if pd.isna(text):
        return np.nan
    return CABA_BARRIOS.get(text, text)


def normalize_operation(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    if 'tempor' in text:
        return 'alquiler_temporal'
    if 'alquiler' in text:
        return 'alquiler'
    if 'venta' in text:
        return 'venta'
    return text.replace(' ', '_')


def normalize_currency(value):
    text = clean_text(value, lower=False)
    if pd.isna(text):
        return np.nan
    t = strip_accents(str(text)).upper().replace('$', 'ARS').strip()
    if t in {'US$', 'U$S', 'USD'}:
        return 'USD'
    if t in {'ARS', 'PESOS', 'PESO'}:
        return 'ARS'
    return t


def replace_invalid_markers(df):
    out = df.copy()
    obj_cols = out.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        stripped = out[col].astype('string').str.strip()
        out.loc[stripped.isin(INVALID_MARKERS), col] = np.nan
    return out


def parse_date_columns(df):
    out = df.copy()
    date_like = [c for c in out.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    for col in date_like:
        out[col] = pd.to_datetime(out[col], errors='coerce', utc=True)
    return out


def add_null_flags(df, exclude=None):
    out = df.copy()
    exclude = set(exclude or [])
    original_cols = [c for c in out.columns if c not in exclude and not c.endswith('_is_null')]
    for col in original_cols:
        if out[col].isna().any():
            out[f'{col}_is_null'] = out[col].isna().astype('int8')
    return out


def to_numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out


def first_existing(df, columns, default=np.nan):
    existing = [c for c in columns if c in df.columns]
    if not existing:
        return pd.Series(default, index=df.index)
    result = df[existing[0]].copy()
    for col in existing[1:]:
        result = result.combine_first(df[col])
    return result

    
def standardize_meli(df, operacion_forzada=None):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'superficie_min_m2', 'superficie_max_m2', 'ambientes_min', 'ambientes_max', 'banios_min', 'banios_max', 'dormitorios_min', 'dormitorios_max', 'comuna', 'precio_m2_min', 'precio_m2_max', 'altura'])
    out['fuente_norm'] = 'mercado_libre'
    out['tipo_operacion_norm'] = operacion_forzada or out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = out.get('barrio', pd.Series(index=out.index, dtype='object')).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_min'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_min_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_min_m2']
    out['id_publicacion'] = out.get('item_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_min_m2', 'superficie_max_m2'])
    out['ambientes_ref'] = first_existing(out, ['ambientes_min', 'ambientes_max'])
    out['banios_ref'] = first_existing(out, ['banios_min', 'banios_max'])
    out['dormitorios_ref'] = first_existing(out, ['dormitorios_min', 'dormitorios_max'])
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_portal(df, fuente_norm):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'superficie_total_m2', 'superficie_cubierta_m2', 'ambientes', 'banios', 'dormitorios', 'comuna', 'precio_m2_calculado', 'latitud', 'longitud', 'altura'])
    out['fuente_norm'] = fuente_norm
    out['tipo_operacion_norm'] = out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = first_existing(out, ['barrio', 'localidad', 'calle']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_calculado'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_total_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_total_m2']
    out['id_publicacion'] = out.get('property_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_total_m2', 'superficie_cubierta_m2'])
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_airbnb(df):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'ambientes', 'banios', 'dormitorios', 'comuna', 'latitud', 'longitud', 'rating', 'reviews', 'capacidad', 'camas'])
    out['fuente_norm'] = 'airbnb'
    out['tipo_operacion_norm'] = 'alquiler_temporal'
    out['barrio_norm'] = first_existing(out, ['barrio_norm', 'barrio']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['id_publicacion'] = out.get('property_id').astype('string')
    out['superficie_ref_m2'] = np.nan
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    out['precio_noche_estimado'] = out['precio']
    out['precio_mensual_estimado'] = np.where(out['precio'].notna(), out['precio'] * 30, np.nan)
    out['precio_m2_ref'] = np.nan
    return out

CANONICAL_COLUMNS = [
    'id_publicacion', 'fuente_norm', 'fuente', 'tipo_operacion_norm', 'tipo_operacion',
    'url', 'fecha_extraccion', 'scraped_at_utc', '_archivo_origen', 'titulo', 'descripcion',
    'tipo_propiedad', 'moneda_norm', 'moneda', 'precio', 'precio_texto', 'precio_m2_ref',
    'precio_noche_estimado', 'precio_mensual_estimado', 'barrio', 'barrio_norm', 'comuna',
    'localidad', 'provincia', 'pais', 'direccion', 'direccion_completa', 'calle', 'altura',
    'latitud', 'longitud', 'superficie_ref_m2', 'superficie_min_m2', 'superficie_max_m2',
    'superficie_total_m2', 'superficie_cubierta_m2', 'ambientes_ref', 'ambientes',
    'ambientes_min', 'ambientes_max', 'dormitorios_ref', 'dormitorios', 'dormitorios_min',
    'dormitorios_max', 'banios_ref', 'banios', 'banios_min', 'banios_max', 'inmobiliaria',
    'seller_id', 'seller_nombre', 'seller_superhost', 'rating', 'reviews', 'apto_credito',
    'balcon', 'balcon_o_patio', 'terraza', 'parrilla', 'pileta', 'cochera',
    'cochera_mencionada', 'amenities_mencionadas', 'precio_sospechoso', 'parse_ok',
    'parse_warnings', 'cantidad_imagenes', 'imagen_principal'
]


def select_canonical(df):
    cols = [c for c in CANONICAL_COLUMNS if c in df.columns]
    return df[cols].copy()

## Revisión de categorías antes de normalizar

In [91]:
cat_vars_relevantes = ['fuente', 'tipo_operacion', 'tipo_propiedad', 'moneda', 'barrio', 'localidad', 'provincia']
for name, df in raw_dfs.items():
    print('\n' + '=' * 100)
    print(name)
    for col in cat_vars_relevantes:
        if col in df.columns:
            print(f'\n-- {col} --')
            display(df[col].astype('string').str.strip().value_counts(dropna=False).head(25).to_frame('conteo'))


df_raw_meli_alq

-- fuente --


,conteo
fuente,
Mercado Libre,11479



-- tipo_operacion --


,conteo
tipo_operacion,
alquiler,11479



-- tipo_propiedad --


...

,conteo
barrio,
Palermo,3220
Recoleta,1321
San Nicolas,570
Belgrano,492
Retiro,470
Monserrat,408
Almagro,373
Balvanera,355
Villa Crespo,292



-- localidad --


,conteo
localidad,
Ciudad Autónoma de Buenos Aires,9596



-- provincia --


,conteo
provincia,
CABA,9596


## Revisión y parseo de fechas

In [92]:
for name, df in raw_dfs.items():
    date_like = [c for c in df.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    print(name, date_like)
    for col in date_like:
        parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
        print(f'  {col}: validas={parsed.notna().sum()} | invalidas/nulas={parsed.isna().sum()}')

df_raw_meli_alq ['scraped_at_utc']
  scraped_at_utc: validas=11479 | invalidas/nulas=0
df_raw_meli_alq_temp ['scraped_at_utc']
  scraped_at_utc: validas=6190 | invalidas/nulas=0
df_raw_meli_vtas ['scraped_at_utc']
  scraped_at_utc: validas=44013 | invalidas/nulas=0
df_raw_argenprop ['scraped_at_utc', 'fecha_publicacion']
  scraped_at_utc: validas=162 | invalidas/nulas=0
  fecha_publicacion: validas=0 | invalidas/nulas=162
df_raw_zonaprop ['scraped_at_utc', 'fecha_publicacion']
  scraped_at_utc: validas=980 | invalidas/nulas=0
  fecha_publicacion: validas=0 | invalidas/nulas=980
df_raw_airbnb_norm ['scraped_at_utc']
  scraped_at_utc: validas=9596 | invalidas/nulas=0


## Normalización por fuente y operación

Airbnb se trata con cuidado porque ya viene normalizado. No se vuelve a parsear desde cero: se conserva su estructura y se crean variables comparables con el resto de las fuentes.

In [93]:
df_preprocesado_meli_alq = standardize_meli(df_raw_meli_alq, operacion_forzada='alquiler')
df_preprocesado_meli_alq_temp = standardize_meli(df_raw_meli_alq_temp, operacion_forzada='alquiler_temporal')
df_preprocesado_meli_vtas = standardize_meli(df_raw_meli_vtas, operacion_forzada='venta')
df_preprocesado_argenprop = standardize_portal(df_raw_argenprop, 'argenprop')
df_preprocesado_zonaprop = standardize_portal(df_raw_zonaprop, 'zonaprop')
df_preprocesado_airbnb_temp = standardize_airbnb(df_raw_airbnb_norm)

preprocesados = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
}
for name, df in preprocesados.items():
    print(name, df.shape)

df_preprocesado_meli_alq (11479, 72)
df_preprocesado_meli_alq_temp (6190, 72)
df_preprocesado_meli_vtas (44013, 72)
df_preprocesado_argenprop (162, 107)
df_preprocesado_zonaprop (980, 107)
df_preprocesado_airbnb_temp (9596, 73)


## Impacto de la transformación de inválidos a nulos

In [94]:
resumen_nulos_pre = []
for name, df in preprocesados.items():
    resumen_nulos_pre.append({
        'dataframe': name,
        'filas': len(df),
        'columnas': df.shape[1],
        'total_nulos': int(df.isna().sum().sum()),
        'pct_celdas_nulas': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2) if df.shape[0] and df.shape[1] else np.nan,
    })
resumen_nulos_pre = pd.DataFrame(resumen_nulos_pre)
display(resumen_nulos_pre)

,dataframe,filas,columnas,total_nulos,pct_celdas_nulas
0,df_preprocesado_meli_alq,11479,72,117771,14.25
1,df_preprocesado_meli_alq_temp,6190,72,64487,14.47
2,df_preprocesado_meli_vtas,44013,72,439061,13.86
3,df_preprocesado_argenprop,162,107,11283,65.09
4,df_preprocesado_zonaprop,980,107,66438,63.36
5,df_preprocesado_airbnb_temp,9596,73,22717,3.24


## Consolidación con columnas canónicas

Se conserva una versión preprocesada por fuente y tipo de operación para evitar que los faltantes estructurales entre portales se confundan con problemas de calidad. Luego se construye una base consolidada con variables comunes y flags de fuente/operación, utilizada para el EDA inicial y comparaciones transversales.

In [95]:
canonical_frames = []
for name, df in preprocesados.items():
    tmp = select_canonical(df)
    tmp['dataset_preprocesado'] = name
    canonical_frames.append(tmp)

df_publicaciones_consolidadas = pd.concat(canonical_frames, ignore_index=True, sort=False)
display(df_publicaciones_consolidadas.head())
print(df_publicaciones_consolidadas.shape)

,id_publicacion,fuente_norm,fuente,tipo_operacion_norm,tipo_operacion,url,fecha_extraccion,scraped_at_utc,_archivo_origen,titulo,tipo_propiedad,moneda_norm,moneda,precio,precio_texto,precio_m2_ref,barrio,barrio_norm,comuna,localidad,provincia,direccion,calle,altura,superficie_ref_m2,superficie_min_m2,superficie_max_m2,ambientes_ref,ambientes_min,ambientes_max,dormitorios_ref,dormitorios_min,dormitorios_max,banios_ref,banios_min,banios_max,inmobiliaria,apto_credito,balcon,terraza,parrilla,pileta,cochera_mencionada,amenities_mencionadas,parse_warnings,imagen_principal,dataset_preprocesado,descripcion,pais,direccion_completa,latitud,longitud,superficie_total_m2,superficie_cubierta_m2,ambientes,dormitorios,banios,seller_id,parse_ok,cantidad_imagenes,precio_noche_estimado,precio_mensual_estimado,seller_nombre,seller_superhost,rating,reviews,balcon_o_patio,cochera,precio_sospechoso
0,MLA2052442697,mercado_libre,Mercado Libre,alquiler,alquiler,https://departamento.mercadolibre.com.ar/MLA-2...,2026-09-03 16:45:14.494669+00:00,2026-09-03 16:45:14.494669+00:00,data\raw\mercadolibre_inmuebles_alquiler\meli_...,Departamento En Alquiler De 4 Ambientes Con De...,departamento,USD,USD,5000.0,US$ 5.000,23.81,Palermo Chico,palermo,14.0,Capital Federal,Capital Federal,"Segui Al 3600, Palermo Chico, Capital Federal",Segui Al,3600.0,210.0,210.0,210.0,4.0,4.0,4.0,NaN,NaN,NaN,3.0,3.0,3.0,Khabbaz Propiedades,0.0,0.0,0,0,0,1.0,1.0,NaN,https://http2.mlstatic.com/D_NQ_NP_2X_926918-M...,df_preprocesado_meli_alq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MLA3906770142,mercado_libre,Mercado Libre,alquiler,alquiler,https://departamento.mercadolibre.com.ar/MLA-3...,2026-09-03 16:45:14.496463+00:00,2026-09-03 16:45:14.496463+00:00,data\raw\mercadolibre_inmuebles_alquiler\meli_...,Alquiler Nuñez 2 Ambientes Con Balcon,departamento,ARS,ARS,1300000.0,$ 1.300.000,20634.92,Núñez,nunez,13.0,Capital Federal,Capital Federal,"Av Libertador Al 8100. Entre Ruiz Huidobro, Nú...",Av Libertador Al 8100. Entre Ruiz Huidobro,NaN,63.0,63.0,63.0,2.0,2.0,2.0,NaN,NaN,NaN,2.0,2.0,2.0,NaN,0.0,1.0,0,0,0,0.0,0.0,NaN,https://http2.mlstatic.com/D_NQ_NP_2X_659510-M...,df_preprocesado_meli_alq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MLA2057533195,mercado_libre,Mercado Libre,alquiler,alquiler,https://departamento.mercadolibre.com.ar/MLA-2...,2026-09-03 16:45:14.497158+00:00,2026-09-03 16:45:14.497158+00:00,data\raw\mercadolibre_inmuebles_alquiler\meli_...,Alquiler Live Hotel 2 Ambientes Full Amenities...,departamento,ARS,ARS,1450000.0,$ 1.450.000,23770.49,Palermo Hollywood,palermo,14.0,Capital Federal,Capital Federal,"Nicaragua Al 6000, Palermo Hollywood, Capital ...",Nicaragua Al,6000.0,61.0,61.0,61.0,2.0,2.0,2.0,NaN,NaN,NaN,1.0,1.0,1.0,NaN,0.0,0.0,0,0,0,0.0,1.0,NaN,https://http2.mlstatic.com/D_NQ_NP_2X_606500-M...,df_preprocesado_meli_alq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MLA3907705666,mercado_libre,Mercado Libre,alquiler,alquiler,https://departamento.mercadolibre.com.ar/MLA-3...,2026-09-03 16:45:14.497799+00:00,2026-09-03 16:45:14.497799+00:00,data\raw\mercadolibre_inmuebles_alquiler\meli_...,Palacio Bellini 3 Dormitorios 2 Baños Toilette...,departamento,USD,USD,3500.0,US$ 3.500,29.17,Palermo,palermo,14.0,Capital Federal,Capital Federal,"República Árabe Siria Al 3000, Palermo, Capita...",República Árabe Siria Al,3000.0,120.0,120.0,120.0,4.0,4.0,4.0,NaN,NaN,NaN,2.0,2.0,2.0,NaN,0.0,0.0,0,0,1,1.0,1.0,NaN,https://http2.mlstatic.com/D_NQ_NP_2X_617430-M...,df_preprocesado_meli_alq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MLA3908849774,mercado_libre,Mercado Libre,alquiler,alquiler,https://departamento.mercadolibre.com.ar/MLA-3...,2026-09-03 16:45:14.498422+00:00,2026-09-03 16:45:14.498422+00:00,data\raw\mercadolibre_inmuebles_alquiler\meli_...,Alquiler Torre Jaramillo Nuñez 2 Dormitorios 2...,

(72420, 69)


## Controles rápidos post-consolidación

## Celdas de decisión: duplicados y campos críticos

Estas celdas dejan explícito el criterio de tratamiento. Por defecto **no se eliminan registros**: se generan reportes y se conserva la trazabilidad. Si el grupo decide aplicar una regla, debe cambiar el parámetro correspondiente y justificarlo en el markdown de cierre.

In [96]:
#parametros de decision conservadores
#cambiar a True solo si se decide aplicar efectivamente el filtro
ELIMINAR_DUPLICADOS_EXACTOS = False
ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS = False

CAMPOS_CRITICOS_CONSOLIDADO = ['id_publicacion', 'fuente_norm', 'tipo_operacion_norm', 'precio', 'moneda_norm', 'barrio_norm']

reporte_decisiones_consolidado = {
    'filas_antes': len(df_publicaciones_consolidadas),
    'duplicados_exactos': int(df_publicaciones_consolidadas.duplicated().sum()),
    'filas_sin_alguno_campos_criticos': int(df_publicaciones_consolidadas[CAMPOS_CRITICOS_CONSOLIDADO].isna().any(axis=1).sum()),
    'eliminar_duplicados_exactos': ELIMINAR_DUPLICADOS_EXACTOS,
    'eliminar_registros_sin_campos_criticos': ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS,
}

if ELIMINAR_DUPLICADOS_EXACTOS:
    df_publicaciones_consolidadas = df_publicaciones_consolidadas.drop_duplicates().copy()

if ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS:
    df_publicaciones_consolidadas = df_publicaciones_consolidadas.dropna(subset=CAMPOS_CRITICOS_CONSOLIDADO).copy()

reporte_decisiones_consolidado['filas_despues'] = len(df_publicaciones_consolidadas)
reporte_decisiones_consolidado = pd.DataFrame([reporte_decisiones_consolidado])
display(reporte_decisiones_consolidado)
reporte_decisiones_consolidado.to_csv(REPORTS_DIR / 'decisiones_filtros_consolidado_v1.csv', index=False)

,filas_antes,duplicados_exactos,filas_sin_alguno_campos_criticos,eliminar_duplicados_exactos,eliminar_registros_sin_campos_criticos,filas_despues
0,72420,0,294,False,False,72420


In [97]:
controles = {
    'filas_por_fuente': df_publicaciones_consolidadas['fuente_norm'].value_counts(dropna=False),
    'filas_por_operacion': df_publicaciones_consolidadas['tipo_operacion_norm'].value_counts(dropna=False),
    'monedas': df_publicaciones_consolidadas['moneda_norm'].value_counts(dropna=False),
    'barrios_top_30': df_publicaciones_consolidadas['barrio_norm'].value_counts(dropna=False).head(30),
}
for titulo, serie in controles.items():
    print('\n' + titulo)
    display(serie.to_frame('conteo'))

campos_criticos = ['id_publicacion', 'fuente_norm', 'tipo_operacion_norm', 'precio', 'moneda_norm', 'barrio_norm', 'superficie_ref_m2', 'precio_m2_ref']
display(df_publicaciones_consolidadas[campos_criticos].isna().mean().mul(100).round(2).to_frame('pct_nulos'))


filas_por_fuente


,conteo
fuente_norm,
mercado_libre,61682
airbnb,9596
zonaprop,980
argenprop,162



filas_por_operacion


,conteo
tipo_operacion_norm,
venta,44580
alquiler_temporal,15786
alquiler,12054



monedas


,conteo
moneda_norm,
USD,62150
ARS,10262
NaN,8



barrios_top_30


,conteo
barrio_norm,
palermo,8874
recoleta,5521
belgrano,4122
caballito,3345
almagro,3275
nunez,3081
balvanera,2912
villa urquiza,2751
puerto madero,2743


,pct_nulos
id_publicacion,0.00
fuente_norm,0.00
tipo_operacion_norm,0.00
precio,0.01
moneda_norm,0.01
barrio_norm,0.39
superficie_ref_m2,13.44
precio_m2_ref,13.45


## Guardado de datasets preprocesados

In [98]:
outputs = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
    'df_publicaciones_consolidadas': df_publicaciones_consolidadas,
}
for name, df in outputs.items():
    path = PROCESSED_DIR / f'{name}_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado:', path.relative_to(REPO_ROOT), df.shape)

Guardado: data\processed\df_preprocesado_meli_alq_v1.csv (11479, 72)
Guardado: data\processed\df_preprocesado_meli_alq_temp_v1.csv (6190, 72)
Guardado: data\processed\df_preprocesado_meli_vtas_v1.csv (44013, 72)
Guardado: data\processed\df_preprocesado_argenprop_v1.csv (162, 107)
Guardado: data\processed\df_preprocesado_zonaprop_v1.csv (980, 107)
Guardado: data\processed\df_preprocesado_airbnb_temp_v1.csv (9596, 73)
Guardado: data\processed\df_publicaciones_consolidadas_v1.csv (72420, 69)


## Diccionario de datos procesado

In [99]:
diccionario_procesado = []
for col in df_publicaciones_consolidadas.columns:
    diccionario_procesado.append({
        'columna': col,
        'tipo_dato': str(df_publicaciones_consolidadas[col].dtype),
        'significado': 'Completar definicion final.',
        'unidad': 'Completar si corresponde.',
        'fuente': 'Base consolidada desde portales inmobiliarios/Airbnb.',
        'transformacion_aplicada': 'Ver notebook 02_normalizacion_y_limpieza.',
        'ejemplo': str(df_publicaciones_consolidadas[col].dropna().iloc[0])[:120] if df_publicaciones_consolidadas[col].notna().any() else np.nan,
    })
diccionario_procesado = pd.DataFrame(diccionario_procesado)
display(diccionario_procesado)
diccionario_procesado.to_csv(PROCESSED_DIR / 'diccionario_datos_procesado_v1.csv', index=False)

,columna,tipo_dato,significado,unidad,fuente,transformacion_aplicada,ejemplo
0,id_publicacion,string,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,MLA2052442697
1,fuente_norm,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,mercado_libre
2,fuente,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,Mercado Libre
3,tipo_operacion_norm,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,alquiler
4,tipo_operacion,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,alquiler
5,url,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,https://departamento.mercadolibre.com.ar/MLA-2...
6,fecha_extraccion,"datetime64[us, UTC]",Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,2026-09-03 16:45:14.494669+00:00
7,scraped_at_utc,"datetime64[us, UTC]",Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,2026-09-03 16:45:14.494669+00:00
8,_archivo_origen,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,data\raw\mercadolibre_inmuebles_alquiler\meli_...
9,titulo,str,Completar definicion final.,Completar si corresponde.,Base consolidada desde portales inmobiliarios/...,Ver notebook 02_normalizacion_y_limpieza.,Departamento En Alquiler De 4 Ambientes Con De...


## Decisiones de limpieza documentadas

In [100]:
decisiones_limpieza = pd.DataFrame([
    {'tema': 'Faltantes estructurales', 'decision': 'Analizar por dataset antes de consolidar', 'justificacion': 'Cada fuente trae columnas distintas; unir todo desde el inicio genera nulos estructurales.'},
    {'tema': 'Airbnb', 'decision': 'Usar archivo normalizado mensual como base prioritaria', 'justificacion': 'Ya contiene barrio, comuna, amenities y variables comparables.'},
    {'tema': 'Barrios', 'decision': 'Crear barrio_norm con limpieza de tildes, aliases y subzonas', 'justificacion': 'Permite cruces territoriales sin perder el valor original.'},
    {'tema': 'Moneda', 'decision': 'Crear moneda_norm', 'justificacion': 'Evita mezclar USD, ARS y representaciones textuales.'},
    {'tema': 'Outliers', 'decision': 'No eliminar en esta etapa; se flaggean en notebook 04', 'justificacion': 'En real estate pueden existir extremos validos.'},
])
display(decisiones_limpieza)
decisiones_limpieza.to_csv(REPORTS_DIR / 'decisiones_limpieza_v1.csv', index=False)

,tema,decision,justificacion
0,Faltantes estructurales,Analizar por dataset antes de consolidar,Cada fuente trae columnas distintas; unir todo...
1,Airbnb,Usar archivo normalizado mensual como base pri...,"Ya contiene barrio, comuna, amenities y variab..."
2,Barrios,"Crear barrio_norm con limpieza de tildes, alia...",Permite cruces territoriales sin perder el val...
3,Moneda,Crear moneda_norm,"Evita mezclar USD, ARS y representaciones text..."
4,Outliers,No eliminar en esta etapa; se flaggean en note...,En real estate pueden existir extremos validos.


## Cierre de limpieza inicial

La salida de este notebook es una familia de datasets preprocesados por fuente/operación y una base consolidada de columnas comunes. El análisis específico de mecanismos de ausencia se separa en el notebook 03; los valores atípicos se tratan en el notebook 04.

In [101]:
print(df_preprocesado_meli_alq[df_preprocesado_meli_alq['inmobiliaria'] == 'SIN_DATO'])

Empty DataFrame
Columns: [item_id, fuente, url, search_url, pagina_origen, scraped_at_utc, titulo, tipo_propiedad, tipo_operacion, moneda, precio, precio_texto, desde, inmobiliaria, direccion, barrio, localidad, provincia, ambientes_min, ambientes_max, banios_min, banios_max, dormitorios_min, dormitorios_max, superficie_min_m2, superficie_max_m2, tipo_superficie, imagen_principal, destacado, emprendimiento, posesion, unidades_disponibles, atributos_texto, texto_tarjeta, parse_warnings, posicion_pagina, slug, calle, altura, comuna, precio_anterior, descuento_pct, precio_m2_min, precio_m2_max, amplitud_superficie_m2, es_rango_superficie, es_rango_ambientes, monoambiente, apto_credito, balcon, terraza, pileta, parrilla, cochera_mencionada, amenities_mencionadas, estado_entrega, cantidad_atributos, completitud_pct, imagen_id, imagen_extension, _archivo_origen, fuente_norm, tipo_operacion_norm, barrio_norm, moneda_norm, precio_m2_ref, id_publicacion, superficie_ref_m2, ambientes_ref, banios

# Tratamiento de duplicados